# Mental Health RAG — Build Vector Index (Colab)

This notebook runs **Stage 3** of the pipeline: embedding chunks and building ChromaDB collections.

**Steps:**
1. Upload `db1_chunks.json` and `db2_chunks.json` from your Mac
2. Embed all chunks with `BAAI/bge-base-en-v1.5`
3. Build two ChromaDB collections
4. Download the `vectordb/` folder back to your Mac

## 1. Install dependencies

In [ ]:
!pip install -q chromadb sentence-transformers

## 2. Upload your chunk files

Run this cell, then click **Choose Files** and upload:
- `data/processed/db1_chunks.json`
- `data/processed/db2_chunks.json`

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

for name in uploaded:
    print(f"Uploaded: {name} ({len(uploaded[name]) / 1024:.0f} KB)")

## 3. Load chunks

In [ ]:
import json

db1_chunks = json.loads(open("db1_chunks.json", encoding="utf-8").read())
db2_chunks = json.loads(open("db2_chunks.json", encoding="utf-8").read())

print(f"DB1 chunks: {len(db1_chunks)}")
print(f"DB2 chunks: {len(db2_chunks)}")
print(f"Total:      {len(db1_chunks) + len(db2_chunks)}")

## 4. Load embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-base-en-v1.5"
print(f"Loading {EMBED_MODEL}...")
model = SentenceTransformer(EMBED_MODEL)
print("Model loaded.")

## 5. Helper functions

In [ ]:
import chromadb
import numpy as np

BATCH_SIZE = 5000


def clean_metadata(meta, allowed_keys):
    """ChromaDB metadata values must be str | int | float | bool."""
    out = {}
    for key in allowed_keys:
        val = meta.get(key)
        if val is None:
            out[key] = ""
        elif isinstance(val, list):
            out[key] = ", ".join(str(v) for v in val)
        else:
            out[key] = val
    return out


def batches(items, size):
    for i in range(0, len(items), size):
        yield items[i : i + size]


def build_collection(chunks, db_key, chroma_dir, coll_name, meta_keys):
    print(f"\n{'='*60}")
    print(f"Building: {coll_name}")
    print(f"{'='*60}")
    print(f"  {len(chunks)} chunks")

    texts     = [c["text"] for c in chunks]
    metadatas = [clean_metadata(c["metadata"], meta_keys) for c in chunks]
    ids       = [f"{db_key}-{i}" for i in range(len(chunks))]

    # Embed
    print(f"  Embedding...")
    embeddings = model.encode(
        texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    print(f"  Embedding shape: {embeddings.shape}")

    # ChromaDB
    os.makedirs(chroma_dir, exist_ok=True)
    client = chromadb.PersistentClient(path=chroma_dir)

    # Drop if exists
    try:
        client.delete_collection(coll_name)
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name=coll_name,
        metadata={"hnsw:space": "cosine"},
    )

    # Batch upsert
    print(f"  Upserting...")
    total = 0
    for batch_ids, batch_embs, batch_docs, batch_metas in zip(
        batches(ids, BATCH_SIZE),
        batches(embeddings.tolist(), BATCH_SIZE),
        batches(texts, BATCH_SIZE),
        batches(metadatas, BATCH_SIZE),
    ):
        collection.upsert(
            ids=batch_ids,
            embeddings=batch_embs,
            documents=batch_docs,
            metadatas=batch_metas,
        )
        total += len(batch_ids)
        print(f"    {total}/{len(chunks)} upserted")

    print(f"  Done. Collection '{coll_name}' has {collection.count()} items.")
    print(f"  Saved to: {chroma_dir}")

## 6. Build DB1 — Clinical collection

In [ ]:
build_collection(
    chunks=db1_chunks,
    db_key="db1",
    chroma_dir="vectordb/clinical",
    coll_name="mental_health_clinical",
    meta_keys=["condition", "section", "source", "source_url", "icd11_code", "chunk_index"],
)

## 7. Build DB2 — Therapy collection

In [ ]:
build_collection(
    chunks=db2_chunks,
    db_key="db2",
    chroma_dir="vectordb/therapy",
    coll_name="mental_health_therapy",
    meta_keys=["technique_name", "modality", "source", "source_url", "chunk_type", "chunk_index"],
)

## 8. Verify collections

In [ ]:
# Quick verification
clinical_client = chromadb.PersistentClient(path="vectordb/clinical")
therapy_client  = chromadb.PersistentClient(path="vectordb/therapy")

clinical_coll = clinical_client.get_collection("mental_health_clinical")
therapy_coll  = therapy_client.get_collection("mental_health_therapy")

print(f"Clinical collection: {clinical_coll.count()} items")
print(f"Therapy collection:  {therapy_coll.count()} items")
print(f"Total vectors:       {clinical_coll.count() + therapy_coll.count()}")

# Test a query
test_query = "symptoms of depression"
test_emb = model.encode([test_query], normalize_embeddings=True).tolist()
results = clinical_coll.query(query_embeddings=test_emb, n_results=3)

print(f"\nTest query: '{test_query}'")
print("Top 3 results:")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"  {i+1}. [{meta.get('condition', '')} / {meta.get('source', '')}] {doc[:100]}...")

## 9. Download the vectordb folder

This zips both collections and downloads them to your Mac.

After downloading, unzip into your project:
```bash
cd /Users/sarramajdoub/Documents/pfa
unzip ~/Downloads/vectordb.zip -d data/
```

In [ ]:
import shutil

shutil.make_archive("vectordb", "zip", ".", "vectordb")
print(f"Archive size: {os.path.getsize('vectordb.zip') / 1024 / 1024:.1f} MB")

files.download("vectordb.zip")

## Done!

After downloading and unzipping, your project should have:
```
data/vectordb/
├── clinical/    ← mental_health_clinical collection
└── therapy/     ← mental_health_therapy collection
```

You can now run `python test_retrieval.py` on your Mac.